In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import cohen_kappa_score

ds = xr.open_dataset("../../NC/compare.nc")
luh3 = xr.open_dataset("../../NC/compareluh3.nc")

luh32005 = luh3.sel(time="2005-01-01")
luh305deg = luh32005.coarsen(lat=2, lon=2, boundary="trim").mean()
# luh305deg = luh305deg.squeeze("time")

In [3]:
def get_dominant(ds, mode, names, time):
    stacks = []
    for name in names:
        if mode == "luh3":
            var = f"{name}"
        else:
            var = f"{mode}_{name}"
        da = ds[var].sel(time=time)
        stacks.append(da)

    data = xr.concat(stacks, dim="class")

    data_filled = data.fillna(0)

    # 计算最大类别 index
    dominant_idx = data_filled.argmax(dim="class")

    # mask：如果所有值都是 0，则设为 NaN
    mask = data_filled.sum(dim="class") == 0
    dominant_idx = dominant_idx.where(~mask)

    return dominant_idx

In [4]:
def kappa_calc_mode(ds,luh3,names):
    time = "2005-01-01"

    dominant_basin=get_dominant(ds,"basin",names,time)
    dominant_region=get_dominant(ds,"region",names,time)
    dominant_luh3=get_dominant(luh3,"luh3",names,time)

    a = dominant_basin.values.flatten()
    b = dominant_region.values.flatten()
    c = dominant_luh3.values.flatten()

    # 去掉 NaN（必须）
    maskbasin = ~np.isnan(a) & ~np.isnan(c)
    maskregion = ~np.isnan(b) & ~np.isnan(c)

    a_valid = a[maskbasin].astype(int)
    c1_valid = c[maskbasin].astype(int)
    b_valid = b[maskregion].astype(int)
    c2_valid = c[maskregion].astype(int)

    kappabasin = cohen_kappa_score(a_valid, c1_valid)
    kapparegion = cohen_kappa_score(b_valid, c2_valid)

    print(f"Kappabasin in {time} = {kappabasin:.4f}")
    print(f"Kapparegion in {time} = {kapparegion:.4f}")
    
    
    return kappabasin,kapparegion

    

In [5]:
names = ["agri", "grassland", "forest"]
k1,k2 = kappa_calc_mode(ds,luh305deg,names)

Kappabasin in 2005-01-01 = 0.7357
Kapparegion in 2005-01-01 = 0.6834


In [6]:
time = "2005-01-01"
names = ["agri", "grassland", "forest"]

dominant_basin  = get_dominant(ds,   "basin",  names, time)
dominant_region = get_dominant(ds,   "region", names, time)
dominant_luh3   = get_dominant(luh305deg, "luh3", names, time)
ds_out = xr.Dataset(
    {
        "dominant_basin":  dominant_basin.astype("float32"),
        "dominant_region": dominant_region.astype("float32"),
        "dominant_luh3":   dominant_luh3.astype("float32"),
    }
)
ds_out.attrs["description"] = "Dominant land-use class (argmax)"
ds_out.attrs["class_0"] = "agri"
ds_out.attrs["class_1"] = "grassland"
ds_out.attrs["class_2"] = "forest"
ds_out.attrs["year"] = 2005
ds_out.to_netcdf("../../NC/dominant_landuse_2005.nc")
